# AeroFlow Components — Data Profiling

This notebook profiles the raw AeroFlow supply chain datasets to understand their structure, data types, missing values and potential data-quality issues before cleaning.

## 1. Import Libraries

In [ ]:
import pandas as pd

In [54]:
parts = pd.read_csv("../data/raw/parts_master.csv")

In [12]:
parts.head()

,part_id,part_family,criticality_class,unit_cost,lead_time_days,supplier_id_primary,supplier_risk_class,is_repairable,shelf_life_days
0,P00001,Electrical,B,2323.45,27,SUP004,Low,No,735.0
1,P00002,Cabin,C,670.84,31,SUP028,Low,No,NaN
2,P00003,Avionics,C,219.44,49,SUP040,Medium,Yes,NaN
3,P00004,Electrical,A,12488.80,33,SUP034,Low,No,NaN
4,P00005,Cabin,C,1319.96,63,SUP024,Medium,No,NaN


In [9]:
parts.shape

(300, 9)

In [10]:
parts.columns

Index(['part_id', 'part_family', 'criticality_class', 'unit_cost',
       'lead_time_days', 'supplier_id_primary', 'supplier_risk_class',
       'is_repairable', 'shelf_life_days'],
      dtype='str')

In [11]:
parts.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   part_id              300 non-null    str    
 1   part_family          300 non-null    str    
 2   criticality_class    300 non-null    str    
 3   unit_cost            300 non-null    float64
 4   lead_time_days       300 non-null    int64  
 5   supplier_id_primary  300 non-null    str    
 6   supplier_risk_class  300 non-null    str    
 7   is_repairable        300 non-null    str    
 8   shelf_life_days      26 non-null     float64
dtypes: float64(2), int64(1), str(6)
memory usage: 21.2 KB


## 2. Missing Value Assessment

In [13]:
parts.isnull().sum()

part_id                  0
part_family              0
criticality_class        0
unit_cost                0
lead_time_days           0
supplier_id_primary      0
supplier_risk_class      0
is_repairable            0
shelf_life_days        274
dtype: int64

In [14]:
parts.groupby("part_family")["shelf_life_days"].count()

part_family
Avionics        0
Cabin           0
Electrical     15
Engine          0
Fasteners       0
Hydraulics     11
LandingGear     0
Structure       0
Name: shelf_life_days, dtype: int64

In [15]:
parts.groupby("part_family").size()

part_family
Avionics       37
Cabin          33
Electrical     41
Engine         32
Fasteners      61
Hydraulics     33
LandingGear    27
Structure      36
dtype: int64

### Investigating Shelf-Life Values

In [16]:
parts[parts["shelf_life_days"].notnull()]

,part_id,part_family,criticality_class,unit_cost,lead_time_days,supplier_id_primary,supplier_risk_class,is_repairable,shelf_life_days
0,P00001,Electrical,B,2323.45,27,SUP004,Low,No,735.0
19,P00020,Hydraulics,C,295.40,46,SUP013,Medium,Yes,950.0
20,P00021,Hydraulics,C,415.37,49,SUP015,Medium,Yes,709.0
36,P00037,Electrical,A,6621.05,37,SUP014,Medium,No,530.0
42,P00043,Electrical,C,1779.33,21,SUP028,Low,No,500.0
57,P00058,Hydraulics,B,1572.49,41,SUP001,Low,Yes,842.0
69,P00070,Electrical,C,747.93,26,SUP027,Medium,No,887.0
73,P00074,Electrical,A,6860.32,22,SUP012,Low,No,1049.0
77,P00078,Electrical,C,1652.36,18,SUP012,Low,No,795.0
111,P00112,Hydraulics,C,226.73,51,SUP025,Medium,Yes,728.0


### Finding: Shelf-Life Data

`shelf_life_days` contains 274 missing values (91.3% of records). Populated values occur only for Electrical and Hydraulics parts, suggesting shelf life is applicable only to selected components rather than being universally required.

The missing values will be retained during profiling and reviewed during the cleaning stage rather than automatically removed or imputed.

## 3. Duplicate Assessment

In [19]:
parts.duplicated().sum()

np.int64(0)

In [21]:
parts["part_id"].duplicated().sum()

np.int64(0)

### Finding: Duplicate Records

No fully duplicated rows were identified in the parts master data. The `part_id` field was also checked independently and all 300 values were unique, confirming it can be used as the unique identifier for each part.

## 4. Categorical Value Assessment

In [22]:
parts["part_family"].unique()

<StringArray>
[ 'Electrical',       'Cabin',    'Avionics',   'Fasteners',   'Structure',
      'Engine',  'Hydraulics', 'LandingGear']
Length: 8, dtype: str

In [23]:
parts["criticality_class"].unique()

<StringArray>
['B', 'C', 'A']
Length: 3, dtype: str

In [24]:
parts["supplier_risk_class"].unique()

<StringArray>
['Low', 'Medium', 'High']
Length: 3, dtype: str

In [25]:
parts["is_repairable"].unique()

<StringArray>
['No', 'Yes']
Length: 2, dtype: str

### Finding: Categorical Values

The categorical fields were reviewed for inconsistent labels and formatting. No issues were identified across `part_family`, `criticality_class`, `supplier_risk_class` or `is_repairable`.

## 5. Numeric Value Assessment

In [26]:
parts.describe()

,unit_cost,lead_time_days,shelf_life_days
count,300.000000,300.000000,26.000000
mean,2109.922333,41.116667,774.076923
std,2572.493113,14.105066,168.920792
min,118.180000,12.000000,437.000000
25%,522.257500,32.000000,650.000000
50%,1155.820000,39.000000,798.500000
75%,2705.557500,49.000000,890.000000
max,18478.000000,100.000000,1063.000000


In [27]:
parts[ parts["lead_time_days"] == parts["lead_time_days"].max()]

,part_id,part_family,criticality_class,unit_cost,lead_time_days,supplier_id_primary,supplier_risk_class,is_repairable,shelf_life_days
293,P00294,Cabin,B,3005.67,100,SUP033,High,No,NaN


In [28]:
parts[ parts["unit_cost"] == parts["unit_cost"].max()]

,part_id,part_family,criticality_class,unit_cost,lead_time_days,supplier_id_primary,supplier_risk_class,is_repairable,shelf_life_days
242,P00243,LandingGear,A,18478.0,20,SUP008,Low,Yes,NaN


## 6. Parts Master Profiling Summary

The `parts_master` dataset contains 300 rows and 9 columns. No duplicate rows were found, and each `part_id` is unique.

The categorical columns were also checked and no inconsistent values were identified.

The main issue found was `shelf_life_days`, which is missing for 274 records (91.3%). The available values only appear for some Electrical and Hydraulics parts, which suggests shelf life may not apply to every part. The missing values were therefore left unchanged for now.

The numeric columns were checked for unusual values. The highest lead time was 100 days and the highest unit cost was 18,478. Both records were reviewed and appeared reasonable, so they were kept in the dataset.

Overall, the dataset is in good condition and only requires limited cleaning.

# Purchase Orders Profiling

The purchase_orders dataset contains purchase order information including suppliers, parts, sites, order quantities and delivery dates. This section checks the structure and quality of the data before cleaning and analysis.

In [3]:
import pandas as pd

In [4]:
purchase_orders = pd.read_csv("../data/raw/purchase_orders.csv")

In [5]:
purchase_orders.head()

,po_id,supplier_id,site_id,part_id,order_date,promised_date,receipt_date,ordered_qty,received_qty
0,PO000001,SUP004,SITE01,P00001,2022-05-02,2022-05-28,2022-05-29,16,16
1,PO000002,SUP004,SITE01,P00001,2022-07-04,2022-07-31,2022-08-02,5,5
2,PO000003,SUP004,SITE01,P00001,2022-10-17,2022-11-12,2022-11-16,5,5
3,PO000004,SUP004,SITE01,P00001,2023-02-06,2023-03-07,2023-03-09,3,3
4,PO000005,SUP004,SITE01,P00001,2023-04-03,2023-04-28,2023-04-29,5,5


In [7]:
purchase_orders.shape   
purchase_orders.columns
purchase_orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 29666 entries, 0 to 29665
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   po_id          29666 non-null  str  
 1   supplier_id    29666 non-null  str  
 2   site_id        29666 non-null  str  
 3   part_id        29666 non-null  str  
 4   order_date     29666 non-null  str  
 5   promised_date  29666 non-null  str  
 6   receipt_date   29666 non-null  str  
 7   ordered_qty    29666 non-null  int64
 8   received_qty   29666 non-null  int64
dtypes: int64(2), str(7)
memory usage: 2.0 MB


## 2. Missing Value Assessment

In [12]:
purchase_orders.isnull().sum()

po_id            0
supplier_id      0
site_id          0
part_id          0
order_date       0
promised_date    0
receipt_date     0
ordered_qty      0
received_qty     0
dtype: int64

## Duplicate assessment

In [14]:
purchase_orders.duplicated().sum()

np.int64(0)

In [16]:
purchase_orders["po_id"].duplicated().sum()   

np.int64(0)

### Finding: Duplicate Records

No fully duplicated rows were found, and all po_id values are unique.

## Date Assessment

In [17]:
purchase_orders["order_date"].min()

'2022-01-03'

In [18]:
purchase_orders["order_date"].max()

'2024-12-23'

In [20]:
pd.to_datetime(purchase_orders["order_date"], errors="coerce").isnull().sum()

np.int64(0)

In [21]:
pd.to_datetime(purchase_orders["promised_date"], errors="coerce").isnull().sum()

np.int64(0)

In [22]:
pd.to_datetime(purchase_orders["receipt_date"], errors="coerce").isnull().sum()

np.int64(0)

In [23]:
(pd.to_datetime(purchase_orders["promised_date"])
    <
    pd.to_datetime(purchase_orders["order_date"])
).sum()

np.int64(0)

In [24]:
(
    pd.to_datetime(purchase_orders["receipt_date"])
    <
    pd.to_datetime(purchase_orders["order_date"])
).sum()

np.int64(0)

### Finding: Date Validation

All three date fields contain valid dates. No promised or receipt dates occur before the corresponding order date, so no date sequence issues were identified.

## Quantity Assessment

In [25]:
purchase_orders[["ordered_qty", "received_qty"]].describe()

,ordered_qty,received_qty
count,29666.000000,29666.000000
mean,14.653105,14.236399
std,13.862931,13.534927
min,1.000000,1.000000
25%,6.000000,6.000000
50%,11.000000,10.000000
75%,19.000000,18.000000
max,263.000000,263.000000


In [28]:
(purchase_orders["received_qty"] < purchase_orders["ordered_qty"]).sum()   

np.int64(3355)

In [29]:
(purchase_orders["received_qty"] > purchase_orders["ordered_qty"]).sum()

np.int64(0)

In [30]:
(purchase_orders["received_qty"] == purchase_orders["ordered_qty"]).sum()

np.int64(26311)

### Finding: Quantity Validation

All purchase orders contained positive ordered and received quantities, with no zero or negative values identified.

Of the 29,666 purchase orders, 3,355 were under-delivered and 26,311 were received in full. No orders were over-delivered.

Under-delivery will be treated as a supplier performance measure rather than a data-quality error.

In [31]:
(
    pd.to_datetime(purchase_orders["receipt_date"])
    >
    pd.to_datetime(purchase_orders["promised_date"])
).sum()

np.int64(16568)

In [32]:
late_delivery_pct = (pd.to_datetime(purchase_orders["receipt_date"]) > pd.to_datetime(purchase_orders["promised_date"])).sum() / len(purchase_orders) * 100

In [34]:
round(late_delivery_pct, 2)

np.float64(55.85)

### Finding: Delivery Performance

16,568 orders (55.85%) were received after their promised date. Further analysis is required to identify which suppliers and parts contribute most to late deliveries.

In [35]:
in_full_pct = (pd.to_datetime(purchase_orders["received_qty"]) == pd.to_datetime(purchase_orders["ordered_qty"])).sum() / len(purchase_orders) * 100

In [36]:
round(in_full_pct, 2)

np.float64(88.69)

In [40]:
otif_orders = (
    (pd.to_datetime(purchase_orders["receipt_date"]) <=
     pd.to_datetime(purchase_orders["promised_date"]))
    &
    (purchase_orders["received_qty"] == purchase_orders["ordered_qty"])
).sum()

In [41]:
otif_orders

np.int64(11789)

In [42]:
otif_pct = (otif_orders / len(purchase_orders)) * 100

In [43]:
round(otif_pct, 2)

np.float64(39.74)

In [44]:
days_difference = (
    pd.to_datetime(purchase_orders["receipt_date"])
    -
    pd.to_datetime(purchase_orders["promised_date"])
)

In [45]:
days_difference.head()

0   1 days
1   2 days
2   4 days
3   2 days
4   1 days
dtype: timedelta64[us]

In [46]:
days_late = days_difference.dt.days

In [47]:
days_late.head()

0    1
1    2
2    4
3    2
4    1
dtype: int64

In [48]:
days_late.describe()

count    29666.000000
mean         1.181723
std          2.989748
min         -3.000000
25%         -1.000000
50%          1.000000
75%          3.000000
max         17.000000
dtype: float64

In [49]:
days_late[days_late > 0]

0        1
1        2
2        4
3        2
4        1
        ..
29649    5
29650    1
29653    3
29656    6
29660    1
Length: 16568, dtype: int64

In [52]:
round(days_late[days_late > 0].mean(), 2)

np.float64(3.33)

### Finding: Delivery Performance

16,568 orders (55.85%) were received after their promised date. Late orders arrived 3.33 days late on average, with the longest delay being 17 days.

Further analysis is required to identify which suppliers, parts and sites contribute most to late deliveries.

In [55]:
purchase_orders[~purchase_orders["part_id"].isin(parts["part_id"])]

,po_id,supplier_id,site_id,part_id,order_date,promised_date,receipt_date,ordered_qty,received_qty


In [56]:
purchase_orders[~purchase_orders["part_id"].isin(parts["part_id"])].shape[0]

0

### Finding: Part ID Validation

All part_id values in purchase_orders matched a corresponding part in parts_master. No unmatched part records were identified.

In [57]:
purchase_orders["supplier_id"].nunique()

40

In [60]:
purchase_orders.groupby("supplier_id").size().sort_values(ascending=False)

supplier_id
SUP027    1311
SUP017    1306
SUP040    1215
SUP030    1171
SUP012    1163
SUP013    1156
SUP006    1096
SUP038     967
SUP010     930
SUP014     843
SUP037     842
SUP036     840
SUP007     831
SUP024     821
SUP031     803
SUP028     793
SUP002     763
SUP022     730
SUP035     699
SUP019     691
SUP029     684
SUP026     680
SUP004     662
SUP001     659
SUP009     649
SUP003     633
SUP033     631
SUP018     607
SUP008     587
SUP034     578
SUP011     565
SUP025     543
SUP021     523
SUP032     441
SUP005     436
SUP020     433
SUP015     413
SUP016     385
SUP039     306
SUP023     280
dtype: int64

In [ ]:
### Finding: Supplier Coverage

The purchase order data contains 40 suppliers. SUP027 has the highest purchase order volume with 1,311 orders, followed by SUP017 with 1,306.

Supplier volume has been reviewed as part of profiling. Detailed supplier performance will be assessed during the analysis stage.

In [62]:
purchase_orders["site_id"].nunique()

6

In [63]:
purchase_orders["site_id"].unique()

<StringArray>
['SITE01', 'SITE02', 'SITE03', 'SITE04', 'SITE05', 'SITE06']
Length: 6, dtype: str

## Purchase Orders Profiling Summary

The purchase_orders dataset contains 29,666 rows and 9 columns covering 40 suppliers and 6 sites.

No missing values, duplicate rows or duplicate po_id values were found. All part_id values matched a corresponding record in parts_master, and the site IDs were consistent.

The three date fields were initially stored as text, but all values could be converted to valid dates. No promised or receipt dates occurred before their corresponding order date.

Quantity checks found no zero or negative values. 26,311 orders were received in full, while 3,355 were under-delivered. No orders were over-delivered.

Delivery performance requires further analysis. 16,568 orders (55.85%) arrived after their promised date, with late orders arriving 3.33 days late on average. The OTIF rate was 39.74%.

Overall, the purchase order data is structurally consistent. The main findings relate to supplier delivery performance rather than data-quality errors.